### 1. Spotify CSV Configuration and Upload

In [1]:
from pathlib import Path
import pandas as pd
import numpy as np

ROOT = Path.cwd().parent  # this notebook lives in notebooks/
SPOTIFY_CSV = ROOT / "data" / "raw" / "spotify_dataset.csv"
OUT = ROOT / "docs" / "evidence" / "profiling"
OUT.mkdir(parents=True, exist_ok=True)

pd.set_option("display.max_columns", 50)
spotify = pd.read_csv(SPOTIFY_CSV)  # read-only: profiling never modifies the data
spotify.head()

,Unnamed: 0,track_id,artists,album_name,track_name,popularity,duration_ms,explicit,danceability,energy,key,loudness,mode,speechiness,acousticness,instrumentalness,liveness,valence,tempo,time_signature,track_genre
0,0,5SuOikwiRyPMVoIQDJUgSV,Gen Hoshino,Comedy,Comedy,73,230666,False,0.676,0.4610,1,-6.746,0,0.1430,0.0322,0.000001,0.3580,0.715,87.917,4,acoustic
1,1,4qPNDBW1i3p13qLCt0Ki3A,Ben Woodward,Ghost (Acoustic),Ghost - Acoustic,55,149610,False,0.420,0.1660,1,-17.235,1,0.0763,0.9240,0.000006,0.1010,0.267,77.489,4,acoustic
2,2,1iJBSr7s7jYXzM8EGcbK5b,Ingrid Michaelson;ZAYN,To Begin Again,To Begin Again,57,210826,False,0.438,0.3590,0,-9.734,1,0.0557,0.2100,0.000000,0.1170,0.120,76.332,4,acoustic
3,3,6lfxq3CG4xtTiEg7opyCyx,Kina Grannis,Crazy Rich Asians (Original Motion Picture Sou...,Can't Help Falling In Love,71,201933,False,0.266,0.0596,0,-18.515,1,0.0363,0.9050,0.000071,0.1320,0.143,181.740,3,acoustic
4,4,5vjLSffimiIP26QG5WcN2K,Chord Overstreet,Hold On,Hold On,82,198853,False,0.618,0.4430,2,-9.681,1,0.0526,0.4690,0.000000,0.0829,0.167,119.949,4,acoustic


### 2. Structure

In [2]:
print("Shape:", spotify.shape)
structure = pd.DataFrame({
    "dtype": spotify.dtypes.astype(str),
    "non_null": spotify.notna().sum(),
    "n_unique": spotify.nunique(),
})
structure.to_csv(OUT / "spotify_structure.csv")
structure

Shape: (114000, 21)


,dtype,non_null,n_unique
Unnamed: 0,int64,114000,114000
track_id,str,114000,89741
artists,str,113999,31437
album_name,str,113999,46589
track_name,str,113999,73608
popularity,int64,114000,101
duration_ms,int64,114000,50697
explicit,bool,114000,2
danceability,float64,114000,1174
energy,float64,114000,2083


### 3. Completeness

In [3]:
missing = spotify.isna().sum().to_frame("missing")
missing["pct"] = (missing["missing"] / len(spotify) * 100).round(3)

text_cols = spotify.select_dtypes("object")
missing["blank_strings"] = text_cols.apply(lambda s: s.str.strip().eq("").sum()).reindex(missing.index).fillna(0).astype(int)

missing.to_csv(OUT / "spotify_completeness.csv")
missing[(missing["missing"] > 0) | (missing["blank_strings"] > 0)]

C:\Users\juans\AppData\Local\Temp\ipykernel_22964\3528857533.py:4: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  text_cols = spotify.select_dtypes("object")


,missing,pct,blank_strings
artists,1,0.001,0
album_name,1,0.001,0
track_name,1,0.001,0


### 4. Uniqueness and Granularity

This part is important:
It tells us whether a `track_id` that appears in multiple genres has different popularity values.

In [4]:
data_cols = [c for c in spotify.columns if not c.startswith("Unnamed")]
print("Fully duplicated rows (ignoring index column):", spotify.duplicated(subset=data_cols).sum())
print("Rows:", len(spotify), "| distinct track_id:", spotify["track_id"].nunique())

genres_per_track = spotify.groupby("track_id")["track_genre"].nunique()
print(genres_per_track.value_counts().sort_index().head(10))

# Do repeated track_ids carry different values in the measures?
measures = ["popularity", "explicit", "danceability", "energy", "valence", "tempo", "duration_ms"]
inconsistent = spotify.groupby("track_id")[measures].nunique().gt(1)
print("track_ids with conflicting values:", inconsistent.any(axis=1).sum())
print(inconsistent.sum())

Fully duplicated rows (ignoring index column): 450
Rows: 114000 | distinct track_id: 89741
track_genre
1    73442
2    11424
3     2955
4     1361
5      431
6      104
7       21
8        2
9        1
Name: count, dtype: int64
track_ids with conflicting values: 720
popularity      720
explicit          0
danceability      0
energy            0
valence           0
tempo             0
duration_ms       0
dtype: int64


### 5. Categorical Content

In [5]:
print(spotify["explicit"].value_counts(dropna=False))
print(spotify["track_genre"].nunique(), "genres")
print(spotify["track_genre"].value_counts().head(15))

artists = spotify["artists"].dropna()
print("Share of rows with several artists (';'):", round(artists.str.contains(";").mean() * 100, 2), "%")
print("Artists with 2+ consecutive spaces:", artists.str.contains(r"\s{2,}").sum())

# Same artist string written with different case/spacing?
norm = artists.str.strip().str.lower()
variants = artists.groupby(norm).nunique()
print("Artist names with case/space variants:", (variants > 1).sum())

explicit
False    104253
True       9747
Name: count, dtype: int64
114 genres
track_genre
acoustic         1000
afrobeat         1000
alt-rock         1000
alternative      1000
ambient          1000
anime            1000
black-metal      1000
bluegrass        1000
blues            1000
brazil           1000
breakbeat        1000
british          1000
cantopop         1000
chicago-house    1000
children         1000
Name: count, dtype: int64
Share of rows with several artists (';'): 26.38 %
Artists with 2+ consecutive spaces: 0
Artist names with case/space variants: 9


### 6. Numeric content (impossible ranges)

In [6]:
spotify.describe().T.to_csv(OUT / "spotify_numeric_summary.csv")

rules = {
    "popularity outside 0-100": ~spotify["popularity"].between(0, 100),
    "duration_ms <= 0": spotify["duration_ms"] <= 0,
    "duration_ms > 1 hour": spotify["duration_ms"] > 3_600_000,
    "danceability outside 0-1": ~spotify["danceability"].between(0, 1),
    "energy outside 0-1": ~spotify["energy"].between(0, 1),
    "valence outside 0-1": ~spotify["valence"].between(0, 1),
    "key outside -1..11": ~spotify["key"].between(-1, 11),
    "mode not in {0,1}": ~spotify["mode"].isin([0, 1]),
    "tempo == 0": spotify["tempo"] == 0,
    "time_signature outside 3-7": ~spotify["time_signature"].between(3, 7),
    "loudness > 0 dB": spotify["loudness"] > 0,
}
numeric_checks = pd.DataFrame({"violations": {k: int(v.sum()) for k, v in rules.items()}})
numeric_checks["pct"] = (numeric_checks["violations"] / len(spotify) * 100).round(3)
numeric_checks.to_csv(OUT / "spotify_numeric_checks.csv")
numeric_checks

,violations,pct
popularity outside 0-100,0,0.000
duration_ms <= 0,1,0.001
duration_ms > 1 hour,16,0.014
danceability outside 0-1,0,0.000
energy outside 0-1,0,0.000
valence outside 0-1,0,0.000
key outside -1..11,0,0.000
"mode not in {0,1}",0,0.000
tempo == 0,157,0.138
time_signature outside 3-7,1136,0.996


### 7. Temporary Content

In [8]:
# The Spotify file has no date or year column, so temporal profiling is not applicable.
# This limitation is documented in the design document (Section 1.1).
print([c for c in spotify.columns if any(k in c.lower() for k in ("date", "year", "release"))])

[]


### 8. Spotify (conflicts, zeros, and duplicate songs)

In [9]:
# How large are the popularity conflicts between repeated track_ids?
g = spotify.groupby("track_id")["popularity"]
spread = (g.max() - g.min())
spread = spread[spread > 0]
print(spread.describe())
spread.to_csv(OUT / "spotify_popularity_conflict_spread.csv")

# popularity == 0 (could mean "no data" rather than "unpopular")
zero_pop = (spotify["popularity"] == 0)
print("popularity == 0:", zero_pop.sum(), f"({zero_pop.mean() * 100:.2f}%)")

# Are the null row and the non-positive duration the same row?
bad = spotify[spotify[["artists", "album_name", "track_name"]].isna().any(axis=1) | (spotify["duration_ms"] <= 0)]
print(bad[["track_id", "artists", "track_name", "duration_ms", "popularity"]])

# Same song under different track_ids (e.g. album vs. single releases)
songs = spotify.drop_duplicates("track_id").groupby(["artists", "track_name"])["track_id"].nunique()
print("artist+title pairs with more than one track_id:", (songs > 1).sum())

# Multi-artist tracks (separator ';')
artists = spotify["artists"].dropna()
print("Rows with several artists:", round(artists.str.contains(";").mean() * 100, 2), "%")
print("Artists with 2+ consecutive spaces:", artists.str.contains(r"\s{2,}").sum())

count    720.000000
mean       1.711111
std        4.688326
min        1.000000
25%        1.000000
50%        1.000000
75%        1.000000
max       44.000000
Name: popularity, dtype: float64
popularity == 0: 16020 (14.05%)
                     track_id artists track_name  duration_ms  popularity
65900  1kR4gIb7nGxHPI3D2ifs59     NaN        NaN            0           0
artist+title pairs with more than one track_id: 4657
Rows with several artists: 26.38 %
Artists with 2+ consecutive spaces: 0


### 9. Connection to the Grammy DB – Info retrieved from the table in the `warehouse-db` container
This is not the data warehouse post-ETL process (the final step, loading, is not). We only read the information directly from the container where the table is stored in PostgreSQL.

In [10]:
import os
from dotenv import load_dotenv
from sqlalchemy import create_engine
from sqlalchemy.engine import URL

load_dotenv(ROOT / ".env")
url = URL.create(
    "postgresql+psycopg2",
    username=os.environ["WH_USER"], password=os.environ["WH_PASSWORD"],
    host="localhost", port=5433, database="grammy_source",
)
engine = create_engine(url)
grammy = pd.read_sql("SELECT * FROM grammy_awards", engine)

print("Shape:", grammy.shape)
g_structure = pd.DataFrame({"dtype": grammy.dtypes.astype(str), "non_null": grammy.notna().sum(), "n_unique": grammy.nunique()})
g_structure.to_csv(OUT / "grammy_structure.csv")
g_structure

Shape: (4810, 11)


,dtype,non_null,n_unique
grammy_id,int64,4810,4810
year,int64,4810,62
title,str,4810,62
published_at,"datetime64[us, UTC]",4810,4
updated_at,"datetime64[us, UTC]",4810,10
category,str,4810,638
nominee,str,4804,4131
artist,str,2970,1658
workers,str,2620,2366
img,str,3443,1463


### 10. Grammy (completeness and uniqueness)

In [11]:
g_missing = grammy.isna().sum().to_frame("missing")
g_missing["pct"] = (g_missing["missing"] / len(grammy) * 100).round(3)
g_missing["blank_strings"] = (grammy.select_dtypes("object").apply(lambda s: s.str.strip().eq("").sum())
                              .reindex(g_missing.index).fillna(0).astype(int))
g_missing.to_csv(OUT / "grammy_completeness.csv")
print(g_missing[(g_missing["missing"] > 0) | (g_missing["blank_strings"] > 0)])

print("Exact duplicates (excluding grammy_id):", grammy.drop(columns="grammy_id").duplicated().sum())
print("winner:", grammy["winner"].value_counts(dropna=False).to_dict())

# Grain check: how many records per (year, category)?
per_cat_year = grammy.groupby(["year", "category"]).size()
print(per_cat_year.describe())
print(per_cat_year.value_counts().sort_index().head(10))
per_cat_year.to_csv(OUT / "grammy_records_per_year_category.csv")

         missing     pct  blank_strings
nominee        6   0.125              0
artist      1840  38.254              0
workers     2190  45.530              0
img         1367  28.420              0
Exact duplicates (excluding grammy_id): 0
winner: {True: 4810}
count    4430.000000
mean        1.085779
std         0.578731
min         1.000000
25%         1.000000
50%         1.000000
75%         1.000000
max         8.000000
dtype: float64
1    4315
2      31
5      79
6       1
8       4
Name: count, dtype: int64


C:\Users\juans\AppData\Local\Temp\ipykernel_22964\3389446932.py:3: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  g_missing["blank_strings"] = (grammy.select_dtypes("object").apply(lambda s: s.str.strip().eq("").sum())


### 11. Grammys (categories, text, and dates)

In [12]:
print("Distinct categories:", grammy["category"].nunique())
cat_norm = grammy["category"].str.strip().str.lower().str.replace(r"\s+", " ", regex=True)
print("Categories with case/space variants:", (grammy.groupby(cat_norm)["category"].nunique() > 1).sum())

nom = grammy["nominee"].dropna()
print("nominee purely numeric:", nom.str.fullmatch(r"\d+").sum(), "of", len(nom))
print(nom.value_counts().head(10))

art = grammy["artist"].dropna()
print("artist null:", grammy["artist"].isna().sum(), "| distinct:", art.nunique())
print("artist with consecutive spaces:", art.str.contains(r"\s{2,}").sum())
print("artist containing ',' ';' or 'feat':", art.str.contains(r",|;|feat", case=False).sum())

# Temporal
print("Years:", grammy["year"].min(), "-", grammy["year"].max())
print("Missing years in range:", sorted(set(range(grammy["year"].min(), grammy["year"].max() + 1)) - set(grammy["year"])))
for c in ["published_at", "updated_at"]:
    print(c, grammy[c].min(), grammy[c].max(), "nulls:", grammy[c].isna().sum())

title_year = grammy["title"].str.extract(r"\((\d{4})\)")[0].astype(float)
print("year != year in title:", (title_year != grammy["year"]).sum(), "| titles without year:", title_year.isna().sum())

Distinct categories: 638
Categories with case/space variants: 8
nominee purely numeric: 12 of 4804
nominee
Steven Epstein                7
Berlioz: Requiem              7
Robert Woods                  7
Bridge Over Troubled Water    7
David Frost                   6
Britten: War Requiem          6
A Taste Of Honey              6
Up, Up And Away               6
Berg: Wozzeck                 5
Blackstar                     5
Name: count, dtype: int64
artist null: 1840 | distinct: 1658
artist with consecutive spaces: 1
artist containing ',' ';' or 'feat': 205
Years: 1958 - 2019
Missing years in range: []
published_at 2017-11-28 08:03:45+00:00 2020-05-19 12:10:28+00:00 nulls: 0
updated_at 2017-11-28 08:03:45+00:00 2020-09-01 19:16:40+00:00 nulls: 0
year != year in title: 0 | titles without year: 0


### 12. Consistency across sources (cross-check by artist)

In [13]:
import unicodedata, re

def norm(s):
    s = unicodedata.normalize("NFKD", str(s)).encode("ascii", "ignore").decode()
    return re.sub(r"\s+", " ", s).strip().lower()

spotify_artists = set(spotify["artists"].dropna().str.split(";").explode().map(norm))
g_art = grammy["artist"].dropna().map(norm)
matched = g_art.isin(spotify_artists)

print("Grammy rows with artist found in Spotify:", round(matched.mean() * 100, 2), "%")
print("Distinct Grammy artists:", g_art.nunique(), "| matched:", g_art[matched].nunique())
unmatched = g_art[~matched].value_counts().head(25)
unmatched.to_csv(OUT / "cross_source_unmatched_top25.csv")
print(unmatched)

Grammy rows with artist found in Spotify: 44.11 %
Distinct Grammy artists: 1653 | matched: 530
artist
(various artists)                  66
u2                                 18
dixie chicks                       12
jimmy sturr                        12
pat metheny group                  10
cece winans                         9
shirley caesar                      9
the manhattan transfer              8
bill cosby                          8
henry mancini                       8
los tigres del norte                7
take 6                              7
steven curtis chapman               6
the black eyed peas                 6
robert plant & alison krauss        6
ricky skaggs & kentucky thunder     6
jose feliciano                      6
amy grant                           6
chet atkins                         6
bobby mcferrin                      6
5th dimension                       6
luis miguel                         5
ziggy marley                        5
dianne reeves           

### 13. v2 normalization and group size

In [14]:
def norm2(s):
    s = unicodedata.normalize("NFKD", str(s)).encode("ascii", "ignore").decode().lower()
    s = s.replace("&", " and ")
    s = re.sub(r"[^a-z0-9 ]", " ", s)
    s = re.sub(r"\s+", " ", s).strip()
    return re.sub(r"^the ", "", s)

sp_keys = set(spotify["artists"].dropna().str.split(";").explode().map(norm2))
g_art2 = grammy["artist"].dropna().map(norm2)
g_art2 = g_art2[g_art2 != "various artists"]

matched2 = g_art2.isin(sp_keys)
print("Rows matched (v2):", round(matched2.mean() * 100, 2), "% of rows with a real artist")
print("Share of ALL Grammy rows:", round(matched2.sum() / len(grammy) * 100, 2), "%")
print("Distinct artists:", g_art2.nunique(), "| matched:", g_art2[matched2].nunique())
print(g_art2[~matched2].value_counts().head(15))

# Size of the groups at track level (a track is Grammy if ANY of its artists is)
tracks = spotify.drop_duplicates("track_id")[["track_id", "artists"]].dropna()
tracks = tracks.assign(artist=tracks["artists"].str.split(";")).explode("artist")
tracks["is_grammy"] = tracks["artist"].map(norm2).isin(set(g_art2))
print(tracks.groupby("track_id")["is_grammy"].any().value_counts())

Rows matched (v2): 46.42 % of rows with a real artist
Share of ALL Grammy rows: 28.02 %
Distinct artists: 1631 | matched: 550
artist
u2                                   18
dixie chicks                         12
jimmy sturr                          12
pat metheny group                    10
cece winans                           9
shirley caesar                        9
manhattan transfer                    8
bill cosby                            8
henry mancini                         8
los tigres del norte                  7
ricky skaggs and kentucky thunder     7
take 6                                7
steven curtis chapman                 6
robert plant and alison krauss        6
jose feliciano                        6
Name: count, dtype: int64
is_grammy
False    82872
True      6868
Name: count, dtype: int64


### 14. Is Spotify a gender-balanced sample?

In [15]:
gc = spotify["track_genre"].value_counts()
print(gc.describe())
print((gc == 1000).sum(), "genres with exactly 1000 rows")

count     114.0
mean     1000.0
std         0.0
min      1000.0
25%      1000.0
50%      1000.0
75%      1000.0
max      1000.0
Name: count, dtype: float64
114 genres with exactly 1000 rows


### 15. Category groups

In [16]:
cat = grammy["category"].str.lower()
fields = ["pop", "rock", "rap", "country", "r&b", "jazz", "classical", "latin", "gospel",
          "reggae", "folk", "dance", "blues", "metal", "album of the year",
          "record of the year", "song of the year", "new artist"]
print({f: int(cat.str.contains(f, regex=False).sum()) for f in fields})
print("No keyword matched:", int((~cat.apply(lambda c: any(f in c for f in fields))).sum()))

{'pop': 352, 'rock': 272, 'rap': 144, 'country': 352, 'r&b': 256, 'jazz': 323, 'classical': 560, 'latin': 150, 'gospel': 264, 'reggae': 40, 'folk': 78, 'dance': 49, 'blues': 127, 'metal': 36, 'album of the year': 84, 'record of the year': 69, 'song of the year': 70, 'new artist': 66}
No keyword matched: 1708
